# Vitrine 3D — Rejouer le ciel américain (tables Gold `flight_replay` et `route_flows`)

Objectif : donner à voir le phénomène étudié depuis J2. On **rejoue en 3D des journées réelles** : chaque vol de l'échantillon devient un avion qui décolle, suit un arc et atterrit à l'heure réelle, coloré selon son retard à l'arrivée. On voit ainsi la **vague de retard de la journée** (7 % à 5 h, 27 % vers 20 h, EDA J2 étape 2) se propager de hub en hub, et la différence entre une journée normale et une journée de tempête.

Deux tables Gold alimentent l'onglet « Ciel 3D » du dashboard (`app/sections/ciel_3d.py`) :

| Table | Grain | Usage |
|---|---|---|
| `flight_replay` | 1 ligne / vol des journées remarquables | animation 3D (Plotly, image par image) |
| `route_flows` | 1 ligne / route (aéroport → aéroport) sur l'année | carte 3D du réseau (pydeck : arcs + colonnes) |

Difficulté technique : les horaires du dataset sont en **heure locale de chaque aéroport**. Un vol Los Angeles → New York qui part à 8 h n'est pas en l'air en même temps qu'un vol Atlanta → Chicago qui part à 8 h. On convertit donc tout en **heure de l'Est**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("V_vitrine_3D").getOrCreate()
sns.set_theme(style="whitegrid", context="notebook")

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
# Vols arrivés : les seuls qui ont un arrival_delay (même périmètre que l'EDA de J2)
spark.sql("""
    SELECT * FROM v_silver
    WHERE NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
""").createOrReplaceTempView("v_arrives")
print("v_silver :", spark.table("v_silver").count(), "vols | v_arrives :", spark.table("v_arrives").count(), "vols arrivés")


def save_gold(df, name, contrat, n_files=1):
    """Vérifie les colonnes du contrat (docs/repartition_J3_J4.md) puis écrit /lakehouse/gold/<name> en Delta."""
    manquantes = [c for c in contrat if c not in df.columns]
    assert not manquantes, f"Colonnes du contrat manquantes pour {name} : {manquantes}"
    path = f"/lakehouse/gold/{name}"
    autres = [c for c in df.columns if c not in contrat]
    (df.select(*contrat, *autres).coalesce(n_files)
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"Table Gold écrite : {path} ({n:,} lignes)")

v_silver : 291023 vols | v_arrives : 285730 vols arrivés


## 1. Fuseaux horaires des États (heure de l'Est commune)

Décalage UTC standard de chaque État et application ou non de l'heure d'été (en 2015 : du 8 mars au 31 octobre inclus). L'Arizona, Hawaï, Porto Rico, les îles Vierges, Guam et les Samoa n'appliquent pas l'heure d'été. Les quelques États à cheval sur deux fuseaux (Floride, Indiana, Kentucky, Tennessee, Texas, Dakota, Nebraska, Kansas, Oregon, Idaho) prennent le fuseau de leurs principaux aéroports : l'erreur est d'une heure au plus pour quelques petits aéroports, sans effet sur la lecture de l'animation.

In [2]:
TZ = {  # État : (décalage UTC standard en heures, applique l'heure d'été)
    **{s: (-5, True) for s in ["CT", "DE", "FL", "GA", "IN", "KY", "MA", "MD", "ME", "MI", "NC", "NH", "NJ", "NY",
                               "OH", "PA", "RI", "SC", "VA", "VT", "WV", "DC"]},
    **{s: (-6, True) for s in ["AL", "AR", "IA", "IL", "KS", "LA", "MN", "MO", "MS", "ND", "NE", "OK", "SD", "TN",
                               "TX", "WI"]},
    **{s: (-7, True) for s in ["CO", "ID", "MT", "NM", "UT", "WY"]},
    "AZ": (-7, False),
    **{s: (-8, True) for s in ["CA", "NV", "OR", "WA"]},
    "AK": (-9, True), "HI": (-10, False),
    "PR": (-4, False), "VI": (-4, False), "GU": (10, False), "AS": (-11, False),
}
spark.createDataFrame([(s, h, d) for s, (h, d) in TZ.items()],
                      "state STRING, utc_offset_h INT, observes_dst BOOLEAN").createOrReplaceTempView("v_tz")

etats = {r[0] for r in spark.sql("SELECT DISTINCT origin_state FROM v_silver WHERE airport_code_valid").collect()}
manquants = sorted(etats - set(TZ))
print(f"{len(etats)} États dans les données, sans fuseau : {manquants or 'aucun'}")
assert not manquants, "Ajoutez ces États au dictionnaire TZ"

54 États dans les données, sans fuseau : aucun


## 2. Choisir les journées à rejouer (Spark SQL, `ROW_NUMBER`)

On calcule le taux de retard et d'annulation de chaque journée (octobre exclu : codes aéroport non valides, donc pas de coordonnées), puis on classe les journées avec des fonctions de fenêtrage :
- la journée la **plus en retard** et la **plus ponctuelle** ;
- la journée avec **le plus d'annulations** (tempête) ;
- une **journée typique** (taux de retard médian, rang du milieu).

In [3]:
jours = spark.sql("""
    WITH quotidien AS (
        SELECT flight_date,
               COUNT(*)                                AS n_vols,
               100 * AVG(CAST(is_cancelled AS INT))    AS taux_annulation,
               -- taux de retard sur les seuls vols arrivés (même définition que v_arrives)
               100 * AVG(CASE WHEN NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
                              THEN CAST(is_delayed_15 AS INT) END) AS taux_retard
        FROM v_silver
        WHERE airport_code_valid
        GROUP BY flight_date
    ),
    classes AS (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY taux_retard DESC)     AS rang_pire,
               ROW_NUMBER() OVER (ORDER BY taux_retard ASC)      AS rang_meilleur,
               ROW_NUMBER() OVER (ORDER BY taux_annulation DESC) AS rang_annulation,
               COUNT(*) OVER ()                                  AS n_jours
        FROM quotidien
    )
    SELECT flight_date, n_vols, ROUND(taux_retard, 1) AS taux_retard, ROUND(taux_annulation, 1) AS taux_annulation,
           CASE WHEN rang_pire = 1                         THEN '1 · Journée la plus en retard'
                WHEN rang_annulation = 1                   THEN '2 · Journée de tempête (record d''annulations)'
                WHEN rang_pire = CAST(n_jours / 2 AS INT)  THEN '3 · Journée typique'
                WHEN rang_meilleur = 1                     THEN '4 · Journée la plus ponctuelle'
           END AS day_label
    FROM classes
""").filter("day_label IS NOT NULL").orderBy("day_label")
jours.show(truncate=False)
assert jours.count() == 4, "Une même journée cumule deux records : ajustez les critères"
jours.createOrReplaceTempView("v_jours")

+-----------+------+-----------+---------------+---------------------------------------------+
|flight_date|n_vols|taux_retard|taux_annulation|day_label                                    |
+-----------+------+-----------+---------------+---------------------------------------------+
|2015-01-04 |825   |49.7       |3.4            |1 · Journée la plus en retard                |
|2015-01-27 |784   |6.3        |19.3           |2 · Journée de tempête (record d'annulations)|
|2015-07-24 |873   |18.0       |0.3            |3 · Journée typique                          |
|2015-09-01 |830   |4.6        |0.4            |4 · Journée la plus ponctuelle               |
+-----------+------+-----------+---------------+---------------------------------------------+



**Lecture**

- Les deux records sont en janvier, ce qui confirme l'effet de l'hiver vu en J2 (étape 2 bis).
- **4 janvier 2015, la plus en retard** : un vol sur deux arrive avec 15 min de retard ou plus, mais seulement 3,4 % d'annulations. C'est le dimanche de retour des fêtes, sur un réseau chargé.
- **27 janvier 2015, la tempête** : c'est le blizzard qui a frappé le nord-est. 19 % des vols sont annulés, mais les vols maintenus partent presque tous à l'heure (6 % de retard seulement, 4 fois moins qu'un jour normal). Les compagnies annulent à l'avance plutôt que de laisser les retards s'accumuler.
- Conséquence pour l'analyse : retard et annulation sont deux réponses différentes à une perturbation. Le taux de retard seul sous-estime les mauvaises journées, d'où le suivi des deux indicateurs dans le dashboard.

## 3. Table Gold `flight_replay` : horaires en heure de l'Est

Pour chaque vol (heures en minutes depuis minuit, heure de l'Est du jour du vol) :
- départ prévu = `scheduled_departure` (hhmm local) converti avec le fuseau de l'État de départ ;
- **décollage** = départ prévu + `departure_delay` + `taxi_out` (roulage) ;
- **atterrissage** = décollage + `air_time`.

Les vols annulés ou déroutés restent dans la table (décollage ou atterrissage NULL) pour les indicateurs, mais ne volent pas dans l'animation.

In [4]:
FLIGHT_REPLAY_COLS = ["flight_date", "day_label", "airline", "airline_name", "flight_number",
                      "origin_airport", "origin_city", "origin_lat", "origin_lon",
                      "destination_airport", "dest_city", "dest_lat", "dest_lon", "distance",
                      "sched_dep_et", "takeoff_et", "landing_et", "departure_delay", "arrival_delay",
                      "is_delayed_15", "is_cancelled", "is_diverted"]

df_replay = spark.sql("""
    WITH base AS (
        SELECT s.*, j.day_label,
               s.flight_date BETWEEN DATE'2015-03-08' AND DATE'2015-10-31' AS heure_ete,
               o.utc_offset_h AS tz_o, o.observes_dst AS dst_o
        FROM v_silver s
        JOIN v_jours j ON s.flight_date = j.flight_date
        JOIN v_tz o    ON s.origin_state = o.state
        WHERE s.airport_code_valid
    ),
    horaires AS (
        SELECT *,
               -- minutes locales -> UTC -> heure de l'Est (UTC-5, UTC-4 en été)
               (CAST(scheduled_departure / 100 AS INT) * 60 + scheduled_departure % 100)
                 - 60 * (tz_o + IF(heure_ete AND dst_o, 1, 0))
                 + 60 * (-5 + IF(heure_ete, 1, 0))            AS sched_dep_et
        FROM base
    )
    SELECT *,
           sched_dep_et + departure_delay + taxi_out          AS takeoff_et,
           sched_dep_et + departure_delay + taxi_out + air_time AS landing_et
    FROM horaires
""")
save_gold(df_replay.select(*FLIGHT_REPLAY_COLS), "flight_replay", FLIGHT_REPLAY_COLS)

# Contrôles : durées de vol plausibles, et pic de trafic en journée (pas à 3 h du matin)
chk = spark.read.format("delta").load("/lakehouse/gold/flight_replay")
chk.groupBy("day_label").agg(
    F.count("*").alias("n_vols"),
    F.sum(F.col("takeoff_et").isNotNull().cast("int")).alias("n_decollages"),
    F.expr("percentile_approx(landing_et - takeoff_et, 0.5)").alias("duree_mediane_min"),
    F.expr("percentile_approx(takeoff_et, array(0.05, 0.5, 0.95))").alias("decollage_p5_p50_p95_min"),
).orderBy("day_label").show(truncate=False)

Table Gold écrite : /lakehouse/gold/flight_replay (3,312 lignes)


+---------------------------------------------+------+------------+-----------------+------------------------+
|day_label                                    |n_vols|n_decollages|duree_mediane_min|decollage_p5_p50_p95_min|
+---------------------------------------------+------+------------+-----------------+------------------------+
|1 · Journée la plus en retard                |825   |797         |97               |[438, 916, 1474]        |
|2 · Journée de tempête (record d'annulations)|784   |633         |79               |[455, 914, 1371]        |
|3 · Journée typique                          |873   |870         |93               |[423, 921, 1400]        |
|4 · Journée la plus ponctuelle               |830   |827         |89               |[424, 897, 1368]        |
+---------------------------------------------+------+------------+-----------------+------------------------+



## 4. Table Gold `route_flows` : le réseau sur l'année

Une ligne par route (aéroport de départ → aéroport d'arrivée) avec au moins 10 vols arrivés dans l'échantillon : volume, taux de retard et médiane. Le dashboard filtre ensuite sur le nombre de vols, car un taux calculé sur 10 vols reste fragile.

In [5]:
ROUTE_FLOWS_COLS = ["origin_airport", "origin_city", "origin_lat", "origin_lon",
                    "destination_airport", "dest_city", "dest_lat", "dest_lon",
                    "distance", "n_vols", "taux_retard", "retard_median"]

df_routes = spark.sql("""
    SELECT origin_airport, FIRST(origin_city) AS origin_city, FIRST(origin_lat) AS origin_lat, FIRST(origin_lon) AS origin_lon,
           destination_airport, FIRST(dest_city) AS dest_city, FIRST(dest_lat) AS dest_lat, FIRST(dest_lon) AS dest_lon,
           CAST(PERCENTILE_APPROX(distance, 0.5) AS INT)        AS distance,
           COUNT(*)                                             AS n_vols,
           ROUND(100 * AVG(CAST(is_delayed_15 AS INT)), 1)      AS taux_retard,
           PERCENTILE_APPROX(arrival_delay, 0.5)                AS retard_median
    FROM v_arrives
    WHERE airport_code_valid
    GROUP BY origin_airport, destination_airport
    HAVING COUNT(*) >= 10
""")
save_gold(df_routes, "route_flows", ROUTE_FLOWS_COLS)
df_routes.orderBy(F.desc("n_vols")).show(8, truncate=False)

Table Gold écrite : /lakehouse/gold/route_flows (3,754 lignes)


+--------------+-------------+----------+----------+-------------------+-------------+--------+----------+--------+------+-----------+-------------+
|origin_airport|origin_city  |origin_lat|origin_lon|destination_airport|dest_city    |dest_lat|dest_lon  |distance|n_vols|taux_retard|retard_median|
+--------------+-------------+----------+----------+-------------------+-------------+--------+----------+--------+------+-----------+-------------+
|LAX           |Los Angeles  |33.94254  |-118.40807|SFO                |San Francisco|37.619  |-122.37484|337     |636   |23.3       |-2           |
|SFO           |San Francisco|37.619    |-122.37484|LAX                |Los Angeles  |33.94254|-118.40807|337     |612   |25.7       |-1           |
|LAX           |Los Angeles  |33.94254  |-118.40807|JFK                |New York     |40.63975|-73.77893 |2475    |608   |19.9       |-7           |
|JFK           |New York     |40.63975  |-73.77893 |LAX                |Los Angeles  |33.94254|-118.40807|

## 5. Aperçu de l'animation 3D

La figure est construite par la même fonction que le dashboard (`app/sections/ciel_3d.py`). Faites pivoter la scène à la souris puis cliquez sur ▶ Lecture.

In [6]:
import sys
sys.path.insert(0, "/workspace/app")
from sections.ciel_3d import build_replay_figure

replay_pd = spark.read.format("delta").load("/lakehouse/gold/flight_replay").toPandas()
jour_pire = replay_pd[replay_pd["day_label"].str.startswith("1")]
fig = build_replay_figure(jour_pire)
print(f"{len(fig.frames)} images d'animation, {len(jour_pire)} vols")
fig.show()

127 images d'animation, 825 vols


**Interprétation** (titres de l'animation relevés toutes les 2 heures)

| Heure (Est) | Journée typique (24/07) | Journée la plus en retard (04/01) | Tempête (27/01) |
|---|---|---|---|
| 08:00 | 60 avions en vol, 2 arriveront en retard | 46 en vol, 8 en retard | 33 en vol, 2 en retard |
| 12:00 | 87 en vol, 13 en retard | 85 en vol, 30 en retard | 63 en vol, 5 en retard |
| 16:00 | 89 en vol, 17 en retard | 79 en vol, 38 en retard | 61 en vol, 5 en retard |
| 20:00 | 98 en vol, 18 en retard | 79 en vol, 52 en retard | 72 en vol, 4 en retard |
| 22:00 | 88 en vol, 21 en retard | 75 en vol, 55 en retard | 59 en vol, 5 en retard |

- **La vague de retard est visible en direct.** Même un jour normal, la part d'avions en retard passe de 3 % à 8 h à 24 % à 22 h. Les retards se propagent d'un vol à l'autre, car le même avion enchaîne plusieurs rotations (EDA J2, étapes 1 et 2).
- **La pire journée amplifie ce mécanisme** : dès midi, plus d'un tiers des avions en vol arriveront en retard, et à 22 h c'est le cas de 3 sur 4. Les hubs les plus touchés sont Dallas (76 % de départs en retard), Chicago O'Hare (75 %), Denver (72 %) et Newark (71 %), c'est-à-dire les grands hubs de correspondance, comme l'a montré l'ACP (axe 1 : taille et congestion).
- **Le jour de tempête, le ciel est plus vide mais ponctuel** : moins d'avions en vol (les annulations), presque aucun en retard. Cela confirme que la tempête se gère par l'annulation, pas par le retard.
- Pour le voyageur, la conclusion est la même qu'en J2 : **partir tôt le matin** est la meilleure protection contre le retard, quelle que soit la journée.